<a href="https://colab.research.google.com/github/indahkhairunnisah-afk/Tugas-2-Teknik-Pengambilan-Sampel-dan-Data-Wrangling/blob/main/Mid_TPSDW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install requests beautifulsoup4

*syntax ini memrintahkan menyiapkan Python agar mampu melakukan pengambilan data (data acquisition) melalui protokol HTTP dan analisis konten web (web content parsing) secara terintegrasi.*

In [9]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
import base64

*Python sedang “mempersiapkan peralatan” untuk bekerja. Requests agar bisa berkomunikasi dengan web, lalu menghadirkan BeautifulSoup sebagai alat untuk mengurai struktur HTML. Setelah itu, Python menyiapkan json untuk mengelola data dalam format JavaScript Object Notation, serta pandas sebagai perangkat analisis dan pengolahan data dalam bentuk tabel. Terakhir, ia membawa base64 untuk melakukan proses encoding dan decoding data biner menjadi teks.*

In [10]:
List_Android = []

for page in range(1, 4):
    url = f"https://myhartono.com/en/android/page-{page}/"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    titles = soup.find_all("a", class_="product-title")
    prices = soup.find_all("span", class_="ty-price-num")

    for title, price in zip(titles, prices):
        List_Android.append({
            "title": title.get_text(strip=True),
            "price": price.get_text(strip=True)
        })

    print(f"Halaman {page} selesai — {len(titles)} produk ditemukan")

Halaman 1 selesai — 24 produk ditemukan
Halaman 2 selesai — 24 produk ditemukan
Halaman 3 selesai — 20 produk ditemukan


*Syntax ini melakukan web scraping untuk mengumpulkan daftar produk Android dari situs myhartono.com. Pertama, menyiapkan wadah kosong bernama List_Android. Lalu, melalui perulangan for page in range(1, 4), Python membuka halaman 1 hingga 3 secara bergantian. Setiap kali membuka halaman, ia mengirim permintaan HTTP dengan requests, kemudian mengurai isi HTML menggunakan BeautifulSoup. Dari hasil parsing, Python mencari elemen judul produk (a dengan kelas product-title) dan harga (span dengan kelas ty-price-num). Selanjutnya, setiap pasangan judul dan harga dimasukkan ke dalam List_Android sebagai dictionary. Terakhir, Python mencetak informasi bahwa halaman tertentu selesai diproses beserta jumlah produk yang ditemukan.*

In [11]:
print(f"Total produk: {len(List_Android)}")
print(List_Android[:3])

Total produk: 68
[{'title': 'SAMSUNG SMARTPHONE GALAXY A57 5G SERIES', 'price': 'Rp'}, {'title': 'SAMSUNG SMARTPHONE GALAXY A37 SERIES', 'price': '7.109.000'}, {'title': 'SAMSUNG SMARTPHONE GALAXY Z FOLD7 5G SERIES', 'price': 'Rp'}]


*print (f"Total produk: {len(List_Android)}") mencetak jumlah total produk yang berhasil dikumpulkan. len(List_Android) menghitung panjang list, yaitu berapa banyak item (produk) yang ada di dalamnya. print(List_Android[:3]) mencetak tiga item pertama dari list List_Android. Notasi [:3] berarti mengambil elemen dari indeks 0 sampai sebelum indeks 3.*

In [12]:
with open("List_Android.json", "w", encoding="utf-8") as f:
    json.dump(List_Android, f, ensure_ascii=False, indent=4)

print("File List_Android.json berhasil disimpan")

File List_Android.json berhasil disimpan


*Syntax ini menyimpan data produk Android yang dikumpulkan ke dalam file JSON dengan format terstruktur dan mudah dibaca.*

In [14]:
with open("List_Android.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(data[:3])

[{'title': 'SAMSUNG SMARTPHONE GALAXY A57 5G SERIES', 'price': 'Rp'}, {'title': 'SAMSUNG SMARTPHONE GALAXY A37 SERIES', 'price': '7.109.000'}, {'title': 'SAMSUNG SMARTPHONE GALAXY Z FOLD7 5G SERIES', 'price': 'Rp'}]


 *Syntax ini membuka file JSON hasil scraping, mengubahnya kembali menjadi objek Python, lalu menampilkan sebagian kecil isi data untuk verifikasi.*

In [15]:
with open("List_Android.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print("EXTRACT - Data awal:")
print(df.head())
print("Shape:", df.shape)

EXTRACT - Data awal:
                                         title      price
0      SAMSUNG SMARTPHONE GALAXY A57 5G SERIES         Rp
1         SAMSUNG SMARTPHONE GALAXY A37 SERIES  7.109.000
2  SAMSUNG SMARTPHONE GALAXY Z FOLD7 5G SERIES         Rp
3  SAMSUNG SMARTPHONE GALAXY Z FLIP7 5G SERIES  6.109.000
4   SAMSUNG SMARTPHONE GALAXY S25 ULTRA SERIES         Rp
Shape: (68, 2)


*kode ini membaca file JSON hasil scraping, mengubahnya menjadi tabel pandas, lalu menampilkan contoh isi dan dimensi tabel.*

*Dalam hasil ekstraksi terlihat bahwa pada kolom price ada beberapa baris hanya berisi "Rp" tanpa angka. Hal ini terjadi karena produk tersebut sedang dalam kondisi promo atau diskon, sehingga harga normal tidak ditampilkan di halaman web. Secara teknis, saat proses scraping, elemen HTML yang diambil (span dengan class ty-price-num) tidak berisi angka harga, melainkan hanya simbol mata uang. Akibatnya, data yang masuk ke DataFrame menjadi "Rp" saja.*

In [16]:
df["title"] = df["title"].str.strip()
df["price"] = (
    df["price"]
    .str.replace(",", "", regex=False)
    .str.replace("Rp", "", regex=False)
    .str.strip()
)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print("TRANSFORM - Data setelah cleaning:")
print(df.head())

TRANSFORM - Data setelah cleaning:
                                         title      price
0      SAMSUNG SMARTPHONE GALAXY A57 5G SERIES           
1         SAMSUNG SMARTPHONE GALAXY A37 SERIES  7.109.000
2  SAMSUNG SMARTPHONE GALAXY Z FOLD7 5G SERIES           
3  SAMSUNG SMARTPHONE GALAXY Z FLIP7 5G SERIES  6.109.000
4   SAMSUNG SMARTPHONE GALAXY S25 ULTRA SERIES           


*kode ini membersihkan teks judul dan harga, menghapus duplikasi, lalu menampilkan data yang sudah rapi dalam bentuk tabel.*

In [17]:
df.to_csv("List_Android.csv", index=False, encoding="utf-8")
print("File List_Android.csv berhasil disimpan")

File List_Android.csv berhasil disimpan


*Syntax ini mengubah tabel hasil scraping menjadi file CSV yang rapi, lalu memberi tahu bahwa file tersebut telah tersimpan.*